## 06_Scoring_and_Export.ipynb
🚀 Offline scoring and export to SQL

In [1]:
# imports 
import pandas as pd
import sqlite3
import joblib  

# load trained model
model = joblib.load("../models/churn_model_final_v1.joblib") 

#  load processed data 
df = pd.read_csv("../data/processed/clean_customer_churn.csv") 

In [2]:
# select X (features) 
exclude_cols = ["CustomerID", "Churn"]
feature_cols = [col for col in df.columns if col not in exclude_cols]

X = df[feature_cols]  # final features for model

In [3]:
#  generate predictions 
# churn probability
df["ChurnProbability"] = model.predict_proba(X)[:, 1]  # probability of churn
# churn prediction 0/1 using threshold 0.4
df["ChurnPrediction"] = (df["ChurnProbability"] > 0.4).astype(int)

# create risk segment 
df["RiskLevel"] = pd.cut(
    df["ChurnProbability"],
    bins=[-0.01, 0.3, 0.7, 1],
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

In [4]:
# connect to SQLite
conn = sqlite3.connect("../data/database/churn_sql.db")  # open/create SQLite db

# export scored data 
df.to_sql("customers", conn, if_exists="replace", index=False)  # write table

# quick check in Python 
print(df.head())  # display first 5 rows

# close connection 
conn.close()  # always close db

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ...  \
0  No phone service             DSL             No  ...   
1                No             DSL            Yes  ...   
2                No             DSL            Yes  ...   
3  No phone service             DSL            Yes  ...   
4                No     Fiber optic             No  ...   

               PaymentMethod MonthlyCharges TotalCharges Churn  \
0           Electronic check          29.85        29.85    No   
1               Mailed check    